In [10]:
import json

with open("../data/corpus.json") as f:
    corpus = json.load(f)
with open("../data/questions.json") as f:
    questions = json.load(f)

print(len(corpus), len(questions))

6000 1000


In [11]:
import re

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z0-9]+", text)

print(tokenize("Do Mitochondria play a role in COVID-19?"))

['do', 'mitochondria', 'play', 'a', 'role', 'in', 'covid', '19']


In [15]:
def naive_search(query, corpus, k=5):
    results = []
    for doc in corpus:
        score = naive_score(query, doc["text"])                       # посчитай счёт для этого документа
        results.append((score, doc["doc_id"]))

    results = sorted(results, reverse=True)
    top = results[:k]
    return [doc_id for score, doc_id in top]   # оставь только id

In [16]:
def naive_score(query, doc_text):
    q_words = set(tokenize(query))
    d_words = set(tokenize(doc_text))
    return len(q_words & d_words)

In [17]:
q = questions[0]
print(q["question"])
print("gold:", q["gold_doc_id"])
print(naive_search(q["question"], corpus))

Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
gold: 21645374
['21645374', '25442819', '18549878', '16401474', '10727545']


In [18]:
doc_words = [set(tokenize(doc["text"])) for doc in corpus]

def naive_search(query, corpus, k=5):
    q_words = set(tokenize(query))
    results = []
    for i, doc in enumerate(corpus):
        score = len(q_words & doc_words[i])
        results.append((score, doc["doc_id"]))
    results = sorted(results, reverse=True)
    return [doc_id for score, doc_id in results[:k]]

print(naive_search(questions[0]["question"], corpus))

['21645374', '25442819', '18549878', '16401474', '10727545']


In [19]:
hits = 0
n = 100

for q in questions[:n]:
    found = naive_search(q["question"], corpus, k=5)
    if q["gold_doc_id"] in found:
        hits += 1

print("Recall@5:", hits / n)

Recall@5: 0.88


In [20]:
id2text = {doc["doc_id"]: doc["text"] for doc in corpus}

misses = []
for q in questions[:100]:
    found = naive_search(q["question"], corpus, k=5)
    if q["gold_doc_id"] not in found:
        misses.append(q)

print(len(misses), "промахов\n")

q = misses[0]
print("ВОПРОС:", q["question"])
print("\nПРАВИЛЬНАЯ СТАТЬЯ:", id2text[q["gold_doc_id"]][:500])
top1 = naive_search(q["question"], corpus, k=1)[0]
print("\nНАЙДЕНО НА 1 МЕСТЕ:", id2text[top1][:500])

12 промахов

ВОПРОС: Double balloon enteroscopy: is it efficacious and safe in a community setting?

ПРАВИЛЬНАЯ СТАТЬЯ: From March 2007 to January 2011, 88 DBE procedures were performed on 66 patients. Indications included evaluation anemia/gastrointestinal bleed, small bowel IBD and dilation of strictures. Video-capsule endoscopy (VCE) was used prior to DBE in 43 of the 66 patients prior to DBE evaluation. The mean age was 62 years. Thirty-two patients were female, 15 were African-American; 44 antegrade and 44 retrograde DBEs were performed. The mean time per antegrade DBE was 107.4±30.0 minutes with a distance 

НАЙДЕНО НА 1 МЕСТЕ: The "health workforce" crisis has led to an increased interest in health professional education, including MPH programs. Recently, it was questioned whether training of mid- to higher level cadres in public health prepared graduates with competencies to strengthen health systems in low- and middle-income countries. Measuring educational impact has been not

In [21]:
import math
from collections import Counter

N = len(corpus)

df = Counter()
for words in doc_words:
    df.update(words)

def idf(word):
    return math.log(1 + (N - df[word] + 0.5) / (df[word] + 0.5))

for w in ["in", "is", "and", "community", "safe", "enteroscopy", "balloon"]:
    print(f"{w:12}  df={df[w]:5}  idf={idf(w):.2f}")

in            df= 5889  idf=0.02
is            df= 3069  idf=0.67
and           df= 5971  idf=0.00
community     df=  125  idf=3.87
safe          df=   31  idf=5.25
enteroscopy   df=    2  idf=7.78
balloon       df=   32  idf=5.22


In [22]:
def idf_search(query, corpus, k=5):
    q_words = set(tokenize(query))
    results = []
    for i, doc in enumerate(corpus):
        common = q_words & doc_words[i]
        score = sum(idf(w) for w in common)
        results.append((score, doc["doc_id"]))
    results = sorted(results, reverse=True)
    return [doc_id for score, doc_id in results[:k]]

hits = 0
for q in questions[:100]:
    if q["gold_doc_id"] in idf_search(q["question"], corpus, k=5):
        hits += 1
print("Recall@5:", hits / 100)


Recall@5: 0.97


In [23]:
doc_tf = [Counter(tokenize(doc["text"])) for doc in corpus]
doc_len = [sum(tf.values()) for tf in doc_tf]
avgdl = sum(doc_len) / N

def bm25_search(query, corpus, k=5, k1=1.5, b=0.75):
    q_words = set(tokenize(query))
    results = []
    for i, doc in enumerate(corpus):
        score = 0.0
        for w in q_words:
            tf = doc_tf[i][w]
            if tf == 0:
                continue
            norm = k1 * (1 - b + b * doc_len[i] / avgdl)
            score += idf(w) * tf * (k1 + 1) / (tf + norm)
        results.append((score, doc["doc_id"]))
    results = sorted(results, reverse=True)
    return [doc_id for score, doc_id in results[:k]]

print(bm25_search(questions[0]["question"], corpus))

['21645374', '25442819', '26538180', '16401474', '18222909']


In [24]:
def evaluate(search_fn, n=300):
    hits1, hits5 = 0, 0
    for q in questions[:n]:
        found = search_fn(q["question"], corpus, k=5)
        if found[0] == q["gold_doc_id"]:
            hits1 += 1
        if q["gold_doc_id"] in found:
            hits5 += 1
    return hits1 / n, hits5 / n

for name, fn in [("naive", naive_search), ("idf", idf_search), ("bm25", bm25_search)]:
    r1, r5 = evaluate(fn)
    print(f"{name:6}  Recall@1={r1:.3f}  Recall@5={r5:.3f}")

naive   Recall@1=0.793  Recall@5=0.883
idf     Recall@1=0.890  Recall@5=0.953
bm25    Recall@1=0.907  Recall@5=0.967


In [25]:
import sys
sys.path.append("../src")

%load_ext autoreload
%autoreload 2

from bm25 import BM25

bm25 = BM25([doc["text"] for doc in corpus])

def bm25_class_search(query, corpus, k=5):
    return [corpus[i]["doc_id"] for score, i in bm25.search(query, k)]

print(bm25_class_search(questions[0]["question"], corpus))
print(evaluate(bm25_class_search))

['21645374', '25442819', '26538180', '16401474', '18222909']
(0.9066666666666666, 0.9666666666666667)
